In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob
import seaborn as sns
import sys
import copy
from tqdm.notebook import tqdm
from numba import jit
from scipy import stats
import networkx as nx
import random
import re
from numba import njit

#plt.style.use('seaborn-deep')
plt.rcParams["text.usetex"] = True
plt.rcParams['text.latex.preamble'] = r'\usepackage{amssymb,amsmath}'

plt.rcParams["figure.figsize"] = 11.7, 8.3
plt.rcParams["figure.dpi"] = 75

plt.rcParams["font.size"] = 24
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Fira Sans", 'PT Sans', 'Open Sans', 'Roboto', 'DejaVu Sans', 'Liberation Sans', 'sans-serif']

plt.rcParams["legend.frameon"] = True
plt.rcParams["legend.fancybox"] = True
plt.rcParams["legend.fontsize"] = "small"

plt.rcParams["lines.linewidth"] = 2.5
plt.rcParams["lines.markersize"] = 14
plt.rcParams["lines.markeredgewidth"] = 2

plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8

# Taken from https://github.com/giotto-ai/giotto-tda/blob/master/examples/classifying_shapes.ipynb

In [3]:
from generate_datasets import make_point_clouds
num_points = 20
num_samples = 10
point_clouds_basic, labels_basic = make_point_clouds(n_samples_per_shape=num_samples, n_points=num_points, noise=0.5)
point_clouds_basic.shape, labels_basic.shape

((30, 400, 3), (30,))

In [4]:
from gtda.plotting import plot_point_cloud

plot_point_cloud(point_clouds_basic[0])

In [5]:
plot_point_cloud(point_clouds_basic[10])

In [6]:
plot_point_cloud(point_clouds_basic[-1])

# RJ some comments:
We need to have the same shape of `all_diagrams` as in the original code so we can use it in the next pipeline.

```
diagrams_basic.shape = (n_examples, X, n_k_values)

where k_values = [0, 1, 2] 
```

X is a some values extracted from PH (but the size does not matter, I think).

Once we will have `all_diagrams` in a right format we could use it in the function:

```
from gtda.diagrams import PersistenceEntropy

persistence_entropy = PersistenceEntropy()

# calculate topological feature matrix
X_basic = persistence_entropy.fit_transform(diagrams_basic)

# expect shape - (n_point_clouds, n_homology_dims)
X_basic.shape

```

And later we can follow what they did.



In [7]:
from pynndescent import NNDescent
import math

def euclidean_distance_3d(p, q):
     return math.sqrt((p[0] - q[0])**2 + (p[1] - q[1])**2 + (p[2] - q[2])**2)

# function to convert point cloud into graph
def convert_pointcloud_to_graph(point_cloud, graph_type, r=None, k=None):
    if graph_type == 'knn':
        if k is None:
            raise ValueError("Parameter 'k' must be provided for 'knn' graph type.")
        # graph construction using knn

        # Compute kNN using NNDescent
        # indices: array where each row contains the indices of the k-nearest neighbors for each point.
        # I put k+1 because the algorithm counts the same node as the nearest
        knnData = NNDescent(point_cloud, metric="euclidean", n_neighbors=k+1, verbose=True)
        indices, distances = knnData.neighbor_graph

        # Create graph
        G = nx.Graph()

        for i in range(len(point_clouds_basic[0])):
            G.add_node(i, pos=tuple(point_cloud[i]))

        # Add edges based on kNN
        for i in range(len(point_cloud)):
            for j in indices[i]:  
                if i!=j: # this is to exclude self loops
                    G.add_edge(i, j)
        return(G)
        
    elif graph_type == 'rgg':
        if r is None:
            raise ValueError("Parameter 'r' must be provided for 'rgg' graph type.")
        # graph construction using rgg

        G = nx.Graph()

        for i in range(len(point_clouds_basic[0])):
            G.add_node(i, pos=tuple(point_cloud[i]))

        # Add edges based on random geometric graph
        for i in range(len(point_cloud)):
            for j in range(len(point_cloud)):  
                if i!=j:
                    if euclidean_distance_3d(point_cloud[i],point_cloud[j]) < r:
                        G.add_edge(i, j)
        return(G)
    
    else:
        raise ValueError("graph_type must be 'rgg' or 'knn'.")


In [8]:
# The function that takes a graph G and computes persistence diagrams with a given edge or node filtration is already created
# it's this one: calculate_persistence_diagram(G, filtration, k_homology)

from filtrations import density_sum_cycles, ollivier_ricci_curvature, edge_betweenness_centrality, vietoris_rips, laplacian_eigenvalues, degrees
from diagrams import calculate_persistence_diagram

filename = "graphs/point_cloud0_knn_4_degree_khomo_1.edgelist"
G = nx.read_edgelist(filename, nodetype=int)  
persistence_diagram_OR = calculate_persistence_diagram(G, ollivier_ricci_curvature, 1)
persistence_diagram_degrees = calculate_persistence_diagram(G, degrees, 1, True) # Use True for node-based filtrations
# density_sum_cycles requires fortran
# persistence_diagram = calculate_persistence_diagram(G, density_sum_cycles, 1)
persistence_diagram_BC = calculate_persistence_diagram(G, edge_betweenness_centrality, 1)
persistence_diagram_laplacian = calculate_persistence_diagram(G, laplacian_eigenvalues, 1, True)
# I had to modify the function generate_shortest_path_distance_matrix in distances.py to ensure node labels are sequential from 0 to len(G)-1
persistence_diagram_VR = calculate_persistence_diagram(G, vietoris_rips, 1, True)


In [11]:
# function that processes all point clouds
import csv

def get_filtration_name(filtration_type):
    if filtration_type == degrees:
        return 'degree'
    if filtration_type == ollivier_ricci_curvature:
        return 'ORC'
    if filtration_type == edge_betweenness_centrality:
        return 'EBC'
    if filtration_type == vietoris_rips:
        return 'VR'
    if filtration_type == laplacian_eigenvalues:
        return 'laplacian'

# k_values = [4, 5, 6, 7, 8, 9, 10, 20, 30]
# r_values = [0.1, 0.3, 0.5, 0.7, 0.9, 1.1, 1.3, 1.5]
def get_persistent_diagrams_from_graphs(point_cloud, num_cloud, graph_type, filtration_type, k_homology, k_values = [4, 5, 6, 7], r_values = [0.1, 0.3]):
    diagrams = []
    filtration_name = get_filtration_name(filtration_type)
    if graph_type == 'knn':
        for k in k_values:
            G = convert_pointcloud_to_graph(point_cloud, 'knn', None, k)
            
            # Save the graph to an edgelist file
            output_filename = f"graphs/point_cloud{num_cloud}_knn_{k}_{filtration_name}_khomo_{k_homology}.edgelist"
            nx.write_edgelist(G, output_filename, data=False) 
            print(f"Graph saved to {output_filename}")
            if filtration_type in [degrees, vietoris_rips, laplacian_eigenvalues]:
                persistence_diagram = calculate_persistence_diagram(G, filtration_type, k_homology, True)
            else:
                persistence_diagram = calculate_persistence_diagram(G, filtration_type, k_homology)
            diagrams.append(persistence_diagram)

            output_filename_diagram = f"diagrams/point_cloud{num_cloud}_knn_{k}_{filtration_name}_khomo_{k_homology}.csv"
            with open(output_filename_diagram, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["birth", "death", "k dimension"]) 
                writer.writerows(persistence_diagram)  
            print(f"Diagram saved to {output_filename_diagram}")
        return diagrams
    
    elif graph_type == 'rgg':
        for r in r_values:
            G = convert_pointcloud_to_graph(point_cloud, 'rgg', r, None)
            
            output_filename = f"graphs/point_cloud{num_cloud}_rgg_{r}_{filtration_name}_khomo_{k_homology}.edgelist"
            nx.write_edgelist(G, output_filename, data=False) 
            print(f"Graph saved to {output_filename}")

            if filtration_type in [degrees, vietoris_rips, laplacian_eigenvalues]:
                persistence_diagram = calculate_persistence_diagram(G, filtration_type, k_homology, True)
            else:
                persistence_diagram = calculate_persistence_diagram(G, filtration_type, k_homology)
            diagrams.append(persistence_diagram)

            output_filename_diagram = f"diagrams/point_cloud{num_cloud}_rgg_{r}_{filtration_name}_khomo_{k_homology}.csv"
            with open(output_filename_diagram, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["birth", "death", "k dimension"]) 
                writer.writerows(persistence_diagram)  
            print(f"Diagram saved to {output_filename_diagram}")
        return diagrams

    else:
        raise ValueError("graph_type must be 'rgg' or 'knn'.")



def get_all_diagrams(point_clouds_basic, filtration_type, graph_type, k_homology_list):
    all_diagrams = []  # List to store diagrams for all point clouds

    for num_cloud in range(len(point_clouds_basic)):
        point_cloud = point_clouds_basic[num_cloud]  
        cloud_diagrams = []  # Collect diagrams for this point cloud
        
        for k_homology in k_homology_list:
            k_diagrams = []  # List to store diagrams for different values of k
            diagrams = get_persistent_diagrams_from_graphs(point_cloud, num_cloud, graph_type, filtration_type, k_homology)
            
            if diagrams:
                for diagram in diagrams:
                    k_diagrams.append(np.array(diagram, dtype=np.float64))  
                cloud_diagrams.append(k_diagrams)  

        all_diagrams.append(cloud_diagrams)
        
    all_diagrams = np.array(all_diagrams, dtype=object)
    all_diagrams.reshape(num_samples, -1, len(k_homology_list))

    return all_diagrams


k_homology_list = [0, 1, 2] 
filtration_type = degrees
graph_type = 'knn'
all_diagrams = get_all_diagrams(point_clouds_basic, filtration_type, graph_type, k_homology_list)

"""
# Add density_sum_cycles
filtrations_list = [degrees, ollivier_ricci_curvature, edge_betweenness_centrality, vietoris_rips, laplacian_eigenvalues]
graph_type_list = ['knn', 'rgg']
# k_homology_list = [0, 1, 2, 3, 4]
# for filtration_type in filtrations_list:
    # for graph_type in graph_type_list: 
"""

Thu Feb 13 00:55:22 2025 Building RP forest with 9 trees
Thu Feb 13 00:55:22 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	Stopping threshold met -- exiting after 2 iterations
Graph saved to graphs/point_cloud0_knn_4_degree_khomo_0.edgelist
Diagram saved to diagrams/point_cloud0_knn_4_degree_khomo_0.csv
Thu Feb 13 00:55:22 2025 Building RP forest with 9 trees
Thu Feb 13 00:55:22 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	Stopping threshold met -- exiting after 2 iterations
Graph saved to graphs/point_cloud0_knn_5_degree_khomo_0.edgelist
Diagram saved to diagrams/point_cloud0_knn_5_degree_khomo_0.csv
Thu Feb 13 00:55:22 2025 Building RP forest with 9 trees
Thu Feb 13 00:55:23 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	Stopping threshold met -- exiting after 2 iterations
Graph saved to graphs/point_cloud0_knn_6_degree_khomo_0.edgelist
Diagram saved to diagrams/point_cloud0_knn_6_degree_khomo_0.csv
Thu Feb 13 00:55:23 2025 Building RP forest with 9 trees
Thu 

"\n# Add density_sum_cycles\nfiltrations_list = [degrees, ollivier_ricci_curvature, edge_betweenness_centrality, vietoris_rips, laplacian_eigenvalues]\ngraph_type_list = ['knn', 'rgg']\n# k_homology_list = [0, 1, 2, 3, 4]\n# for filtration_type in filtrations_list:\n    # for graph_type in graph_type_list: \n"

In [21]:
for i, cloud_diagrams in enumerate(all_diagrams):
    print(f"Point cloud {i + 1}:")
    for k, diagrams in enumerate(cloud_diagrams):
        print(f"  k={k_homology_list[k]}: {len(diagrams)} diagrams")  

print(all_diagrams)
all_diagrams.shape

Point cloud 1:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 2:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 3:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 4:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 5:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 6:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 7:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 8:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 9:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 10:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 11:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 12:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 13:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 14:
  k=0: 4 diagrams
  k=1: 4 diagrams
  k=2: 4 diagrams
Point cloud 15:
  k=0: 4 diag

(30, 3, 4)

In [13]:
from gtda.homology import VietorisRipsPersistence

# Track connected components, loops, and voids
homology_dimensions = [0, 1, 2]

# Collapse edges to speed up H2 persistence calculation!
persistence = VietorisRipsPersistence(
    metric="euclidean",
    homology_dimensions=homology_dimensions,
    n_jobs=6,
    collapse_edges=True,
)

diagrams_basic = persistence.fit_transform(point_clouds_basic)
print(diagrams_basic)

[[[0.         0.00286574 0.        ]
  [0.         0.00307521 0.        ]
  [0.         0.00486446 0.        ]
  ...
  [0.152101   0.152101   2.        ]
  [0.152101   0.152101   2.        ]
  [0.152101   0.152101   2.        ]]

 [[0.         0.00477758 0.        ]
  [0.         0.00637809 0.        ]
  [0.         0.00780919 0.        ]
  ...
  [0.152101   0.152101   2.        ]
  [0.152101   0.152101   2.        ]
  [0.152101   0.152101   2.        ]]

 [[0.         0.00673785 0.        ]
  [0.         0.00676173 0.        ]
  [0.         0.00695781 0.        ]
  ...
  [0.152101   0.152101   2.        ]
  [0.152101   0.152101   2.        ]
  [0.152101   0.152101   2.        ]]

 ...

 [[0.         0.06993221 0.        ]
  [0.         0.07405581 0.        ]
  [0.         0.0800097  0.        ]
  ...
  [0.152101   0.152101   2.        ]
  [0.152101   0.152101   2.        ]
  [0.152101   0.152101   2.        ]]

 [[0.         0.03464153 0.        ]
  [0.         0.05104267 0.        ]


In [14]:
point_clouds_basic.shape

(30, 400, 3)

In [15]:
diagrams_basic.shape

(30, 597, 3)

In [16]:
print(all_diagrams)
all_diagrams.shape

[[[array([[ 4., inf,  0.],
          [ 4.,  7.,  0.],
          [ 4.,  7.,  0.],
          [ 4.,  6.,  0.],
          [ 4.,  6.,  0.],
          [ 4.,  6.,  0.],
          [ 4.,  6.,  0.],
          [ 4.,  6.,  0.],
          [ 4.,  6.,  0.],
          [ 4.,  6.,  0.],
          [ 4.,  6.,  0.],
          [ 4.,  6.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
          [ 4.,  5.,  0.],
 

(30, 3, 4)

In [20]:
from gtda.diagrams import PersistenceEntropy

persistence_entropy = PersistenceEntropy()

# calculate topological feature matrix
X_basic = persistence_entropy.fit_transform(diagrams_basic)

# expect shape - (n_point_clouds, n_homology_dims)
X_basic.shape

(30, 3)

In [11]:
plot_point_cloud(X_basic)

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(oob_score=True)
rf.fit(X_basic, labels_basic)

print(f"OOB score: {rf.oob_score_:.3f}")

OOB score: 1.000


In [13]:
from gtda.pipeline import Pipeline

steps = [
    ("persistence", VietorisRipsPersistence(metric="euclidean", homology_dimensions=homology_dimensions, n_jobs=6)),
    ("entropy", PersistenceEntropy()),
    ("model", RandomForestClassifier(oob_score=True)),
]

pipeline = Pipeline(steps)

In [14]:
pipeline.fit(point_clouds_basic, labels_basic)

Pipeline(steps=[('persistence',
                 VietorisRipsPersistence(homology_dimensions=[0, 1, 2],
                                         n_jobs=6)),
                ('entropy', PersistenceEntropy()),
                ('model', RandomForestClassifier(oob_score=True))])

In [15]:
pipeline["model"].oob_score_

1.0

---
---

In [16]:
from openml.datasets.functions import get_dataset

df = get_dataset('shapes').get_data(dataset_format='dataframe')[0]
df.head()

,x,y,z,target
0,0.341007,0.318606,0.096725,human_arms_out9
1,0.329226,0.421601,0.056749,human_arms_out9
2,0.446869,0.648674,0.124090,human_arms_out9
3,0.314729,0.217860,0.070847,human_arms_out9
4,0.426678,0.919195,0.047609,human_arms_out9


In [17]:
import openml

In [18]:
plot_point_cloud(df.query('target == "biplane0"')[["x", "y", "z"]].values)

In [19]:
import numpy as np

point_clouds = np.asarray(
    [
        df.query("target == @shape")[["x", "y", "z"]].values
        for shape in df["target"].unique()
    ]
)
point_clouds.shape

(40, 400, 3)

In [20]:
persistence = VietorisRipsPersistence(
    metric="euclidean",
    homology_dimensions=homology_dimensions,
    n_jobs=6,
    collapse_edges=True,
)
persistence_diagrams = persistence.fit_transform(point_clouds)

In [21]:
# Index - (human_arms_out, 0), (vase, 10), (dining_chair, 20), (biplane, 30)
index = 30
plot_diagram(persistence_diagrams[index])

In [22]:
persistence_entropy = PersistenceEntropy(normalize=True)
# Calculate topological feature matrix
X = persistence_entropy.fit_transform(persistence_diagrams)
# Visualise feature matrix
plot_point_cloud(X)

In [23]:
labels = np.zeros(40)
labels[10:20] = 1
labels[20:30] = 2
labels[30:] = 3

rf = RandomForestClassifier(oob_score=True, random_state=42)
rf.fit(X, labels)
rf.oob_score_

0.6